# Strands Decider 2B — Authenticated Remote Inference Server

**DevFlow AI — AWS Agent Experience Hackathon**

This notebook runs the official **Strands Decider 2B** (`StrandsAgents/strands-decider-2B-qwen3.5-v1-2610`) model server with:
1. **Automatic Device Detection**: Runs on GPU (CUDA) if available; gracefully falls back to CPU mode (`--device cpu`) if Colab assigns a CPU instance.
2. **Bearer Token Authentication**: Protects the server behind a FastAPI reverse proxy so the public tunnel cannot be abused by unauthorized parties.
3. **Zero-Setup Secure Tunnel**: Exposes the authenticated proxy via Cloudflare Tunnel (`pycloudflared`) without requiring an account or port forwarding.

### Step 1: Hardware & Device Verification
Checks if an NVIDIA GPU (e.g. Tesla T4) is attached. If no GPU is available, detects CPU and configures the server accordingly.

In [ ]:
import torch
import psutil
import os

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"Total System RAM: {ram_gb:.2f} GB")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    DEVICE = "cuda"
    print(f"[+] GPU Available: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
    print("    Mode: High-Performance GPU Inference (~20-115ms latency)")
else:
    DEVICE = "cpu"
    print("[-] GPU Not Available (Colab CPU Runtime)")
    print(f"    Mode: CPU-Only Inference ({ram_gb:.2f} GB RAM is sufficient for 2B weights)")
    print("    Latency will be higher (~400-1200ms), but fully functional.")

### Step 2: Install Official Packages & Dependencies
Installs `strands-decider`, `fastapi`, `uvicorn`, `httpx`, and `pycloudflared`.

In [ ]:
!pip install -q strands-decider fastapi uvicorn httpx pycloudflared
print("[+] Packages installed successfully!")

### Step 3: Configure Authentication Token
A secure random Bearer token is generated to protect your remote model endpoint.

In [ ]:
import secrets

# Generate or set your secret API key
API_KEY = secrets.token_hex(16)
print(f"=======================================================")
print(f"YOUR STRANDS DECIDER API KEY:")
print(f"{API_KEY}")
print(f"=======================================================")
print("Keep this key handy. You will paste it into your DevFlow AI environment.")

### Step 4: Write Secure Reverse Proxy Script
This FastAPI proxy sits on port 8000 and requires `Authorization: Bearer <API_KEY>` for all incoming requests. Unauthorized requests receive `401 Unauthorized`. Valid requests are forwarded locally to `strands-decider` on port 8099.

In [ ]:
proxy_code = f'''
from fastapi import FastAPI, Header, HTTPException, Request, Response
import httpx
import uvicorn

app = FastAPI(title="Strands Decider Auth Proxy")
SECRET_KEY = "{API_KEY}"
DECIDER_INTERNAL_URL = "http://127.0.0.1:8099"

def verify_token(authorization: str | None = Header(None)):
    if not authorization:
        raise HTTPException(status_code=401, detail="Missing Authorization header")
    parts = authorization.split()
    if len(parts) != 2 or parts[0].lower() != "bearer" or parts[1] != SECRET_KEY:
        raise HTTPException(status_code=401, detail="Invalid Bearer token")

@app.get("/health")
async def proxy_health(authorization: str | None = Header(None)):
    verify_token(authorization)
    async with httpx.AsyncClient() as client:
        try:
            res = await client.get(f"{{DECIDER_INTERNAL_URL}}/health", timeout=5.0)
            return res.json()
        except Exception as e:
            return {{"online": False, "error": str(e)}}

@app.post("/v1/systemone")
async def proxy_systemone(request: Request, authorization: str | None = Header(None)):
    verify_token(authorization)
    body = await request.body()
    async with httpx.AsyncClient() as client:
        try:
            res = await client.post(
                f"{{DECIDER_INTERNAL_URL}}/v1/systemone",
                content=body,
                headers={{"Content-Type": "application/json"}},
                timeout=30.0,
            )
            return Response(content=res.content, status_code=res.status_code, media_type="application/json")
        except Exception as e:
            raise HTTPException(status_code=502, detail=f"Internal Decider error: {{e}}")

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)
'''

with open("auth_proxy.py", "w") as f:
    f.write(proxy_code)
print("[+] Created auth_proxy.py successfully.")

### Step 5: Launch Strands Decider & Secure Tunnel
Starts `strands-decider serve` on port 8099, the auth proxy on port 8000, and opens an encrypted Cloudflare Tunnel to port 8000.

In [ ]:
import subprocess
import time
from pycloudflared import try_cloudflare

# 1. Start strands-decider server
model_checkpoint = "StrandsAgents/strands-decider-2B-qwen3.5-v1-2610"
device_flag = f"--device {DEVICE}"
print(f"[*] Starting strands-decider on internal port 8099 (device: {DEVICE})...")
decider_proc = subprocess.Popen([
    "strands-decider", "serve", model_checkpoint,
    "--host", "127.0.0.1", "--port", "8099",
    "--device", DEVICE
])

# Wait 15 seconds for initial weight loading
print("[*] Loading model weights into memory...")
time.sleep(15)

# 2. Start FastAPI auth proxy on port 8000
print("[*] Starting Authenticated Reverse Proxy on port 8000...")
proxy_proc = subprocess.Popen(["python", "auth_proxy.py"])
time.sleep(3)

# 3. Expose port 8000 via Cloudflare Tunnel
print("[*] Creating encrypted Cloudflare Tunnel to port 8000...")
tunnel_url = try_cloudflare(port=8000).tunnel_url

print("\n" + "=" * 75)
print("STRANDS DECIDER 2B SERVER IS LIVE AND SECURED!")
print("=" * 75)
print(f"Public Tunnel URL : {tunnel_url}")
print(f"Bearer API Key    : {API_KEY}")
print("=" * 75)
print("\nRun these exact commands in your Windows PowerShell terminal:")
print(f'$env:STRANDS_DECIDER_URL = "{tunnel_url}"')
print(f'$env:STRANDS_DECIDER_API_KEY = "{API_KEY}"')
print('E:\\devflow_venv\\Scripts\\python.exe verify_live_e2e.py')
print("=" * 75)

### Step 6: Test Authenticated Inference (In-Notebook Verification)
Tests a real tool-selection decision using the authenticated tunnel.

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

# 1. Test /health
health_res = requests.get(f"{tunnel_url}/health", headers=headers, timeout=10)
print("Health Check Response:", health_res.json())

# 2. Test /v1/systemone with a real tool-selection query
payload = {
    "decision_type": "tool_selection",
    "prompt": "Why is my pytest asserting assert 1 == 2?",
    "choices": ["explain_code", "debug_error", "analyze_test", "search_docs"],
    "context": "FAILED test_app.py::test_calc - AssertionError: 1 != 2"
}

decision_res = requests.post(f"{tunnel_url}/v1/systemone", json=payload, headers=headers, timeout=20)
print("\nDecision Response from Live Model:")
print(decision_res.json())